# EDA рекомендательной системы

Запускайте из каталога `generate_data` (или поправьте `ROOT` в первой ячейке). Нужны: миграции, данные в БД (`import_projects_dataset`, `seed_recommendation_demo`), зависимости из `2025_hse_dsa/requirements.txt` и скачанная модель MiniLM.

In [1]:
# pip install matplotlib pandas 

In [2]:
!pwd

/Users/user/Desktop/Учеба/cursach/generate_data


In [1]:
import os
import sys

import django

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '2025_hse_dsa', 'src'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'config.settings')
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')  # для Jupyter

os.environ['POSTGRES_HOST'] = 'localhost'
os.environ['POSTGRES_PORT'] = '5433'

django.setup()

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from django.contrib.auth import get_user_model
from django.db.models import Count

from apps.projects.models import Application, Project, Tag
from apps.projects.services import (
    COLLAB_WEIGHT,
    GRADES_WEIGHT,
    SEMANTIC_WEIGHT,
    TAG_WEIGHT,
    _collaborative_scores,
    _grades_scores,
    _semantic_scores,
    _tag_scores,
    get_recommended_projects,
)

User = get_user_model()

In [5]:
import json
import random
from pathlib import Path

from apps.projects.synthetic_student_profiles import format_llm_prompt, plan_rows

STUDENTS_N = 150
SEED = 42

rng = random.Random(SEED)
tags = list(Tag.objects.order_by('pk'))
student_plan_rows = plan_rows(tags, rng, STUDENTS_N)

DATA_DIR = Path(ROOT) / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
PLAN_PATH = DATA_DIR / 'student_plan.json'
with open(PLAN_PATH, 'w', encoding='utf-8') as f:
    json.dump(student_plan_rows, f, ensure_ascii=False, indent=2)

print('План студентов:', PLAN_PATH)
print('--- Промпт для LLM (скопируйте ниже целиком) ---')
print(format_llm_prompt(student_plan_rows))

План студентов: /Users/user/Desktop/Учеба/cursach/2025_hse_dsa/src/data/student_plan.json
--- Промпт для LLM (скопируйте ниже целиком) ---
Ты генерируешь правдоподобные профили студентов ВШЭ для демо рекомендательной системы проектов.

Правила:
- Пиши на русском.
- Поля bio и cover_letter должны быть согласованы с указанными тегами (интересы в тематике DS/IT/бизнес и т.д.).
- bio: 2–4 предложения о себе, опыте и целях.
- cover_letter: 3–6 предложений, почему интересны проектные работы и как совпадают с тегами.
- Не выдумывай конкретные имена компаний/лабораторий, если не уверен; можно обобщать («крупный маркетплейс», «IT-компания»).

Верни строго один JSON-массив из ровно 150 объектов (без markdown, без комментариев вне JSON):
[
  {"index": <int>, "bio": "<строка>", "cover_letter": "<строка>"},
  ...
]

Индексы index должны совпасть с перечисленными ниже.

Студенты:
- index=1, программа=Прикладная математика и информатика, кампус=Нижний Новгород, курс=1, уровень=бакалавриат, теги: Anal

In [ ]:
# Вставьте ответ LLM одним JSON-массивом (можно с обёрткой ```json).
from apps.projects.synthetic_student_profiles import parse_llm_json_array

LLM_RESPONSE = """
"""

profiles = parse_llm_json_array(LLM_RESPONSE)
if len(profiles) != STUDENTS_N:
    raise ValueError(f'Ожидали {STUDENTS_N} объектов, получили {len(profiles)}')

PROFILES_PATH = DATA_DIR / 'synthetic_student_profiles.json'
with open(PROFILES_PATH, 'w', encoding='utf-8') as f:
    json.dump(profiles, f, ensure_ascii=False, indent=2)
print('Сохранено:', PROFILES_PATH)

Сохранено: /Users/user/Desktop/Учеба/cursach/2025_hse_dsa/src/data/synthetic_student_profiles.json


In [ ]:
from pathlib import Path

from django.core.management import call_command

profiles_file = DATA_DIR / 'synthetic_student_profiles.json'
kw = dict(students=STUDENTS_N, seed=SEED, max_applications=3)
if profiles_file.is_file():
    kw['profiles_json'] = str(profiles_file)
call_command('seed_recommendation_demo', **kw)
# это норм, чутка подредачил вывод потом

Готово: создано 0 студентов и 288 заявок.
Эмбеддинги обновлены: тегов 23, проектов 205.
Готово.
